In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from gsplat import rasterization
from depth_anything_3.api import DepthAnything3

# ---- env & config -----------------------------------------------------------
os.environ.setdefault("HF_HOME",       "/work/scratch/nmeurer/hf_cache")
os.environ.setdefault("HF_HUB_CACHE",  "/work/scratch/nmeurer/hf_cache/hub")
os.environ.setdefault("XDG_CACHE_HOME","/work/scratch/nmeurer/cache")

DATA_ROOT       = Path("/cluster/courses/cil/monocular-depth-estimation/train")
TEACHER_MODEL   = "depth-anything/DA3-GIANT-1.1"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- load model + single inference pass (geometry + Gaussians) -------------
model = DepthAnything3.from_pretrained(TEACHER_MODEL, cache_dir=os.environ["HF_HUB_CACHE"]).to(device).eval()

In [ ]:
# =============================================================================
# Self-contained: Method B (provenance-based confidence masking) end-to-end.
#
# Loads a sample, runs DA3 once with the GS head, builds 4 toe-in cameras,
# splats RGB / confidence / depth into each view, masks by confidence trust,
# and visualizes everything.
# =============================================================================

PROCESS_RES     = 560
USE_TOE_IN      = True
DELTA_FRACTION  = 0.05      # shift magnitude as fraction of scene depth
P_THRESH        = 50.0      # percentile for trust mask
SAMPLE_ID       = "train_022453"      # set e.g. "train_001324" to pin; else random

# ---- pick a sample ----------------------------------------------------------
rgbs = sorted(DATA_ROOT.glob("*_rgb.png"))
if SAMPLE_ID is None:
    rgb_path = rgbs[np.random.randint(len(rgbs))]
else:
    rgb_path = DATA_ROOT / f"{SAMPLE_ID}_rgb.png"
depth_path = rgb_path.parent / rgb_path.name.replace("_rgb.png", "_depth.npy")
print(f"Sample: {rgb_path.stem.replace('_rgb','')}")

prediction = model.inference(
    image=[str(rgb_path)],
    infer_gs=True,
    process_res=PROCESS_RES,
)

# ---- unpack camera + Gaussians ---------------------------------------------
extr_w2c = prediction.extrinsics[0]        # (3, 4) world-to-camera
K        = prediction.intrinsics[0]        # (3, 3)
depth    = prediction.depth[0]             # (H, W)
conf_in  = prediction.conf[0]              # (H, W)
H, W     = depth.shape

# Camera world-axis vectors
R    = extr_w2c[:, :3]
t    = extr_w2c[:,  3]
R_c2w_in   = R.T
c_w        = -R.T @ t
right_w    = R_c2w_in[:, 0]
down_w     = R_c2w_in[:, 1]
forward_w  = R_c2w_in[:, 2]
up_w       = -down_w

# Scene scale from depth median (robust to sky/outliers)
valid = (conf_in > np.percentile(conf_in, 30)) & np.isfinite(depth) & (depth > 0)
scene_depth = float(np.median(depth[valid]))

# Gaussians: squeeze batch, set up SH color
g          = prediction.gaussians
means      = g.means[0].cuda().contiguous()                              # (P, 3)
quats      = g.rotations[0].cuda().contiguous()                          # (P, 4)
scales     = g.scales[0].cuda().contiguous()                             # (P, 3)
opacities  = g.opacities[0].cuda().contiguous()                          # (P,)
colors_sh  = g.harmonics[0].permute(0, 2, 1).cuda().contiguous()         # (P, 9, 3)
sh_degree  = 2

P = means.shape[0]
assert conf_in.size == P, f"Mismatch: {conf_in.size} pixels vs {P} Gaussians."
conf_per_g = torch.from_numpy(conf_in.reshape(-1)).float().cuda()        # (P,)
conf_color = conf_per_g.unsqueeze(-1).expand(-1, 3).contiguous()          # (P, 3)

# ---- build cameras: input view + 4 toe-in shifts ---------------------------
def look_at_opencv(eye, target, up_hint):
    f = target - eye; f /= np.linalg.norm(f)
    r = np.cross(up_hint, f); r /= np.linalg.norm(r)
    d = np.cross(f, r);       d /= np.linalg.norm(d)
    return np.stack([r, d, f], axis=1)

def c2w_to_w2c_4x4(R_c2w, c):
    Rw = R_c2w.T
    M = np.eye(4, dtype=np.float64); M[:3,:3] = Rw; M[:3,3] = -Rw @ c
    return M

up_hint  = down_w
T_target = c_w + scene_depth * forward_w
delta    = DELTA_FRACTION * scene_depth

shifts = [("left", -delta, 0.0), ("right", delta, 0.0),
          ("up",    0.0,  delta), ("down",  0.0, -delta)]

cameras = []
# input view first
M_in = np.eye(4, dtype=np.float64); M_in[:3,:] = extr_w2c
cameras.append(("input", M_in.astype(np.float32)))
# four toe-in views
for label, dx, dy in shifts:
    c_new = c_w + dx * right_w + dy * up_w
    R_c2w_new = look_at_opencv(c_new, T_target, up_hint) if USE_TOE_IN else R_c2w_in
    cameras.append((label, c2w_to_w2c_4x4(R_c2w_new, c_new).astype(np.float32)))

# ---- splatting helpers ------------------------------------------------------
K_t = torch.from_numpy(K.astype(np.float32)).cuda().unsqueeze(0)         # (1,3,3)

def splat_rgb(extr_np):
    img, alpha, _ = rasterization(
        means=means, quats=quats, scales=scales, opacities=opacities,
        colors=colors_sh, sh_degree=sh_degree,
        viewmats=torch.from_numpy(extr_np).cuda().unsqueeze(0),
        Ks=K_t, width=W, height=H,
    )
    return img[0].clamp(0, 1), alpha[0, ..., 0]

def splat_scalar(extr_np, per_g_3ch):
    img, alpha, _ = rasterization(
        means=means, quats=quats, scales=scales, opacities=opacities,
        colors=per_g_3ch, sh_degree=None,
        viewmats=torch.from_numpy(extr_np).cuda().unsqueeze(0),
        Ks=K_t, width=W, height=H,
    )
    return img[0, ..., 0], alpha[0, ..., 0]

def gaussian_depth_in_camera(extr_np):
    """z-coordinate of each Gaussian in the given camera's frame."""
    Rc = torch.from_numpy(extr_np[:3,:3]).float().cuda()
    tc = torch.from_numpy(extr_np[:3, 3]).float().cuda()
    z  = (means @ Rc.T)[:, 2] + tc[2]
    return z.unsqueeze(-1).expand(-1, 3).contiguous()

# ---- run for every camera ---------------------------------------------------
results = []
for label, extr in cameras:
    rgb, _              = splat_rgb(extr)
    conf_img, alpha_c   = splat_scalar(extr, conf_color)
    depth_img, _        = splat_scalar(extr, gaussian_depth_in_camera(extr))

    conf_norm = conf_img / alpha_c.clamp(min=1e-6)
    trust     = (alpha_c * conf_norm).detach().cpu().numpy()
    rgb_np    = (rgb.detach().cpu().numpy() * 255).astype(np.uint8)
    depth_np  = depth_img.detach().cpu().numpy()

    thresh = np.percentile(trust, P_THRESH)
    mask   = trust >= thresh
    results.append(dict(label=label, rgb=rgb_np, depth=depth_np,
                        trust=trust, mask=mask, kept=mask.mean()))

# ---- Figure 1: input image + GT depth + DA3 predicted depth ----------------
rgb_orig = np.array(Image.open(rgb_path).convert("RGB"))
gt_depth = np.load(depth_path).astype(np.float32)
gt_vis   = np.where(gt_depth > 0, gt_depth, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb_orig);              axes[0].set_title("Input RGB");          axes[0].axis("off")
im1 = axes[1].imshow(gt_vis, cmap="viridis");       axes[1].set_title("GT depth");           axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
im2 = axes[2].imshow(depth, cmap="viridis");        axes[2].set_title("DA3 first-pass depth");axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
fig.suptitle(rgb_path.stem.replace("_rgb",""))
plt.tight_layout(); plt.show()

# ---- Figure 2: per-view RGB | reprojected depth | masked depth -------------
n = len(results)
fig, axes = plt.subplots(n, 3, figsize=(15, 4*n))
for i, r in enumerate(results):
    axes[i,0].imshow(r["rgb"])
    axes[i,0].set_title(f"{r['label']}: rendered RGB"); axes[i,0].axis("off")

    im_d = axes[i,1].imshow(r["depth"], cmap="viridis")
    axes[i,1].set_title(f"{r['label']}: reprojected depth"); axes[i,1].axis("off")
    plt.colorbar(im_d, ax=axes[i,1], fraction=0.046, pad=0.04)

    masked = np.where(r["mask"], r["depth"], np.nan)
    im_m = axes[i,2].imshow(masked, cmap="viridis")
    axes[i,2].set_title(f"{r['label']}: masked (p{P_THRESH:.0f}, kept {r['kept']*100:.0f}%)")
    axes[i,2].axis("off")
    plt.colorbar(im_m, ax=axes[i,2], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()